# Task 4: Model Optimization

Perform hyperparameter tuning for the SVM model using Grid Search and Cross-Validation to identify the best parameters (`C` and `gamma`).

**Goal**: Optimize SVM performance for cancer classification.

## 1. Initialize Project Environment

In [1]:
import logging
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)

print(f"Python {sys.version}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]


## 2. Define Configuration Parameters

In [2]:
@dataclass
class TaskConfig:
    handle: str
    artifacts_dir: Path = Path("artifacts")
    cv_folds: int = 5
    random_state: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["artifacts_dir"] = str(info["artifacts_dir"])
        return info


CONFIG = TaskConfig(handle="rbals")
CONFIG.describe()

{'handle': 'rbals',
 'artifacts_dir': 'artifacts',
 'cv_folds': 5,
 'random_state': 42}

## 3. Implement Core Functionality

In [3]:
def optimize_svm(config: TaskConfig):
    # Load processed data
    X_train = pd.read_csv(config.artifacts_dir / "task1_X_train_scaled.csv")
    y_train = pd.read_csv(config.artifacts_dir / "task1_y_train.csv").values.flatten()

    # Define parameter grid
    param_grid = {
        "C": [0.1, 1, 10, 100],
        "gamma": [1, 0.1, 0.01, 0.001],
        "kernel": ["rbf"],
    }

    # Initialize GridSearch
    grid = GridSearchCV(
        SVC(random_state=config.random_state),
        param_grid,
        refit=True,
        verbose=1,
        cv=config.cv_folds,
    )

    # Fit
    grid.fit(X_train, y_train)

    logging.info(f"Best Parameters: {grid.best_params_}")
    logging.info(f"Best Estimator Score: {grid.best_score_}")

    return grid.best_params_, grid.cv_results_


BEST_PARAMS, CV_RESULTS = optimize_svm(CONFIG)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


2026-02-01 16:32:06,249 | INFO | Best Parameters: {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
2026-02-01 16:32:06,250 | INFO | Best Estimator Score: 0.9037037037037037


## 4. Validate with Unit Tests

In [4]:
assert "C" in BEST_PARAMS, "Optimization failed to find C"
assert "gamma" in BEST_PARAMS, "Optimization failed to find gamma"
print("[OK] Validation passed.")

[OK] Validation passed.


## 5. Export Results

In [5]:
EXPORT_DIR = CONFIG.artifacts_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save best parameters
params_df = pd.DataFrame([BEST_PARAMS])
params_out = EXPORT_DIR / "task4_optimized_svm_params.csv"
params_df.to_csv(params_out, index=False)

# Save full CV results
cv_results_df = pd.DataFrame(CV_RESULTS)
cv_out = EXPORT_DIR / "task4_svm_cv_results.csv"
cv_results_df.to_csv(cv_out, index=False)

print(f"[OK] Optimized parameters and CV results saved to artifacts/")

[OK] Optimized parameters and CV results saved to artifacts/
